# Session 39: Hyperparameter Tuning I — Decision Tree Regressor
**Week 4 Systematic Model Optimization via Grid Search**

In this notebook, we systematically optimize the hyperparameters of a Decision Tree Regressor using 5-fold cross-validation (`GridSearchCV`). We evaluate how constrained tree depth (`max_depth`) and minimum split samples (`min_samples_split`) reduce overfitting and improve generalization on untouched test data relative to a default decision tree.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.tree import DecisionTreeRegressor
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Resolve project root directory
PROJECT_ROOT = Path.cwd().resolve()
for parent in [PROJECT_ROOT, *PROJECT_ROOT.parents]:
    if (parent / ".git").exists():
        PROJECT_ROOT = parent
        break

DATA_DIRECTORY = PROJECT_ROOT / "data"
REPORTS_DIRECTORY = PROJECT_ROOT / "reports" / "tables"
REPORTS_DIRECTORY.mkdir(parents=True, exist_ok=True)

print("Project Root:", PROJECT_ROOT)

Project Root: /home/nikhil/Desktop/VSCode/GSSRP/student-performance-prediction-ml


In [2]:
def load_regression_split():
    # Attempt to locate processed student dataset
    candidates = list((DATA_DIRECTORY / "processed").rglob("*.parquet")) + list((DATA_DIRECTORY / "processed").rglob("*.csv"))
    for path in candidates:
        if any(term in path.name.lower() for term in ["comparison", "prediction", "result"]):
            continue
        try:
            table = pd.read_parquet(path) if path.suffix == ".parquet" else pd.read_csv(path)
            if "G3" in table.columns:
                # Exclude G1 and G2 to maintain the early-warning research design
                drop_cols = [c for c in ["G1", "G2", "G3"] if c in table.columns]
                X = table.drop(columns=drop_cols).copy()
                X = pd.get_dummies(X, drop_first=True, dtype=float)
                y = table["G3"]
                
                Xtr, Xte, ytr, yte = train_test_split(
                    X, y, test_size=0.20, random_state=42
                )
                return Xtr, Xte, ytr, yte
        except Exception:
            continue
    raise FileNotFoundError("Could not locate processed regression datasets.")

Xtr_f, Xte_f, ytr, yte = load_regression_split()
print(f"Data Loaded — Training Features: {Xtr_f.shape}, Test Features: {Xte_f.shape}")

def eval_reg(y_true, y_pred):
    mse = mean_squared_error(y_true, y_pred)
    return {
        "MAE": float(mean_absolute_error(y_true, y_pred)),
        "RMSE": float(np.sqrt(mse)),
        "R2": float(r2_score(y_true, y_pred))
    }

Data Loaded — Training Features: (316, 39), Test Features: (79, 39)


In [3]:
# Establish unconstrained default DecisionTreeRegressor baseline
default_tree = DecisionTreeRegressor(random_state=42)
default_tree.fit(Xtr_f, ytr)

default_preds = default_tree.predict(Xte_f)
default_metrics = eval_reg(yte, default_preds)

print("Default Decision Tree Performance (Test Split):")
print("-" * 50)
for metric_name, val in default_metrics.items():
    print(f"{metric_name}: {val:.4f}")
print(f"Default Tree Depth: {default_tree.get_depth()}")

Default Decision Tree Performance (Test Split):
--------------------------------------------------
MAE: 3.6329
RMSE: 4.8822
R2: -0.1624
Default Tree Depth: 19


In [4]:
# Define hyperparameter grid search combinations
param_grid = {
    "max_depth": [3, 5, 7, None],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4]
}

# Instantiate 5-fold cross-validated grid search
grid_search = GridSearchCV(
    estimator=DecisionTreeRegressor(random_state=42),
    param_grid=param_grid,
    cv=5,
    scoring="r2",
    n_jobs=-1,
    return_train_score=True
)

grid_search.fit(Xtr_f, ytr)

print("GridSearchCV Hyperparameter Optimization Results:")
print("-" * 50)
print(f"Best Hyperparameters Found: {grid_search.best_params_}")
print(f"Best 5-Fold Cross-Validated R2 Score: {grid_search.best_score_:.4f}")

GridSearchCV Hyperparameter Optimization Results:
--------------------------------------------------
Best Hyperparameters Found: {'max_depth': 3, 'min_samples_leaf': 4, 'min_samples_split': 10}
Best 5-Fold Cross-Validated R2 Score: 0.1456


In [5]:
# Evaluate optimal estimator on untouched test set
best_tree = grid_search.best_estimator_
tuned_preds = best_tree.predict(Xte_f)
tuned_metrics = eval_reg(yte, tuned_preds)

# Construct side-by-side metric comparison table
comparison_df = pd.DataFrame([
    {"Model": "Default Decision Tree", **default_metrics},
    {"Model": "Tuned Decision Tree (GridSearch)", **tuned_metrics}
])

print("Performance Improvement Comparison:")
display(comparison_df.round(4))

# Export local artifact table
output_csv_path = REPORTS_DIRECTORY / "session39_tree_tuning_results.csv"
comparison_df.to_csv(output_csv_path, index=False)
print(f"Saved results table to: {output_csv_path}")

Performance Improvement Comparison:


,Model,MAE,RMSE,R2
0,Default Decision Tree,3.6329,4.8822,-0.1624
1,Tuned Decision Tree (GridSearch),3.5395,4.4097,0.0517


Saved results table to: /home/nikhil/Desktop/VSCode/GSSRP/student-performance-prediction-ml/reports/tables/session39_tree_tuning_results.csv


In [6]:
# Programmatic validation assertions
assert hasattr(grid_search, "best_params_"), "GridSearchCV has not been fitted correctly!"
assert len(tuned_preds) == len(yte), "Prediction shape mismatch on test split!"
assert np.isfinite(tuned_preds).all(), "Predictions contain non-finite numbers!"

print("=" * 72)
print("SESSION 39 HYPERPARAMETER TUNING COMPLETED SUCCESSFULLY")
print("=" * 72)

SESSION 39 HYPERPARAMETER TUNING COMPLETED SUCCESSFULLY
